In [63]:
import openseespywin as ops
import opsvis as opsv 
import numpy as np
import ipywidgets as widgets
import os
import matplotlib.pyplot as plt
import math
import opstool as opst
import eurocodepy as ecpy
import time as tt
# import openseespy.postprocessing.Get_Rendering as opsplt

In [64]:
import openseespy.opensees as ops

#Soil Element Properties
elasto_mat_tag = 1
thickness = 1.0
type = "PlaneStrain"

# Soil Material properties
E = 200e6         # Elastic modulus in Pa
nu = 0.3          # Poisson's ratio
rho = 0.0         # Density

# Storage for created nodes and elements
created_nodes = []
created_elements = []
block_registry = {}

def generate_block(start_x, start_y, length, height, node_offset, elem_offset, mat_tag, box_width, box_height, block_name="block"):
    num_x = int(length / box_width)
    num_y = int(height / box_height)

    def block_node_id(i, j):
        return node_offset + j * (num_x + 1) + i + 1

    block_nodes = []
    block_elements = []

    # Create nodes
    for j in range(num_y + 1):
        for i in range(num_x + 1):
            nid = block_node_id(i, j)
            x = start_x + i * box_width
            y = start_y + j * box_height
            ops.node(nid, x, y)
            created_nodes.append((nid, x, y))
            block_nodes.append(nid)

    # Create elements
    eid = elem_offset
    for j in range(num_y):
        for i in range(num_x):
            n1 = block_node_id(i, j)
            n2 = block_node_id(i + 1, j)
            n3 = block_node_id(i + 1, j + 1)
            n4 = block_node_id(i, j + 1)

            ops.element("quad", eid, n1, n2, n3, n4, thickness, type, mat_tag)
            created_elements.append((eid, n1, n2, n3, n4))
            block_elements.append(eid)
            eid += 1

    # Register block
    block_registry[block_name] = {
        "nodes": block_nodes,
        "elements": block_elements,
    }

    return (node_offset + (num_x + 1) * (num_y + 1), elem_offset + num_x * num_y)

# Start model
ops.wipe()
ops.model("Basic", "-ndm", 2, "-ndf", 2)

# Define material
ops.nDMaterial("ElasticIsotropic", elasto_mat_tag, E, nu, rho)

# Define contact material
contact_mat = 2
ops.nDMaterial("ContactMaterial2D", contact_mat, 0.1, 1000.0, 0.0, 0.0)

# Generate one block starting from node 1 and element 1
node_offset = 0
elem_offset = 1
node_offset, elem_offset = generate_block(-30.0, 0.0, 30.0, 9, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="OAE_Soil_block")
node_offset, elem_offset = generate_block(-30.0, 9.5, 30.0, 9, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="F1_Soil_block")
node_offset, elem_offset = generate_block(-30.0, 19.0, 23.5, 10, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="F1A_Soil_block")
node_offset, elem_offset = generate_block(-30.0, 29.5, 25.0, 2, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="Estaurine_Soil_block")
node_offset, elem_offset = generate_block(-30.0, 32.0, 30.0, 5.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="EstaurineA_Soil_block")
node_offset, elem_offset = generate_block(-30.0, 38.0, 30.0, 2.0, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="Fill_Soil_block")

In [65]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

4619


In [66]:
# Add custom nodes for segment lining
interface_at_lining = [
[-1.500, 18.565],[-3.500, 19.000],[-4.000, 19.000],[-4.500, 19.000],[-5.000, 19.000],[-5.500, 19.000],
[-6.000, 19.000],[-6.000, 19.500],[-6.000, 20.000],[-6.000, 20.500],[-6.000, 21.000],[-6.000, 21.500],[-6.000, 22.000],
[-6.242, 22.500],[-6.381, 22.985],[-6.439, 23.500],[-6.386, 26.453],[-6.000, 28.000],[-6.000, 28.500],[-6.000, 29.000],
[-5.500, 29.000],[-4.500, 30.000],[-4.500, 30.500],[-4.500, 31.000],[-4.500, 31.500],[-4.000, 31.500],[-3.500, 31.500],
[-3.000, 31.500],[-2.500, 31.500],[-2.000, 31.500],[-1.500, 31.500],[-4.000, 31.000],[-3.500, 31.000],[-3.000, 31.000],
[-3.485, 30.778],[-3.841, 30.778],[-3.679, 30.710],[-4.000, 30.500],[-5.694, 28.654],[-5.500, 19.500],[-5.500, 20.000],
[-5.500, 20.500],[-5.500, 21.000],[-5.000, 19.500],[-4.500, 19.500],[-4.000, 19.500],[-5.000, 20.000],[-4.500, 20.000],
[-5.000, 20.500],[-5.212, 20.798],[-5.388, 21.024],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(interface_at_lining):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["interface_nodes_at_lining"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [67]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

4670


In [68]:
# Add custom nodes for segment lining
lining_nodes = [
[ 0.000, 31.300],[-0.499, 31.280],[-0.995, 31.220],[-1.485, 31.122],[-1.966, 30.985],[-2.434, 30.810],[-2.887, 30.599],[-3.322, 30.352],[-3.736, 30.072],[-4.127, 29.760],
[-4.491, 29.417],[-4.827, 29.047],[-5.133, 28.652],[-5.406, 28.233],[-5.646, 27.795],[-5.849, 27.338],[-6.016, 26.867],[-6.146, 26.384],[-6.236, 25.893],[-6.287, 25.395],[-6.299, 24.896],
[-6.271, 24.396],[-6.203, 23.901],[-6.096, 23.413],[-5.951, 22.934],[-5.769, 22.469],[-5.550, 22.020],[-5.297, 21.589],[-5.009, 21.180],[-4.691, 20.795],[-4.343, 20.436],[-3.967, 20.106],
[-3.567, 19.807],[-3.144, 19.540],[-2.701, 19.308],[-2.241, 19.112],[-1.768, 18.953],[-1.283, 18.832],[-0.789, 18.749],[-0.291, 18.706],[ 0.000, 18.700],
]

start_id = 5000
segment_node_ids = []

for i, (x, y) in enumerate(lining_nodes):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["lining_nodes"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [69]:
master_node_ids = block_registry["lining_nodes"]["nodes"]
print(master_node_ids)

[5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010, 5011, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5027, 5028, 5029, 5030, 5031, 5032, 5033, 5034, 5035, 5036, 5037, 5038, 5039, 5040]


In [70]:
slave_node_setA = [
[-0.500, 31.500],[-1.000, 31.500],[-1.326, 31.414],[-1.820, 31.292],[-2.286, 31.137],[-2.796, 30.923],[-3.237, 30.694],[-3.679, 30.419],
[-4.089, 30.116],[-4.462, 29.794],[-4.827, 29.426],[-5.149, 29.047],[-5.457, 28.622],[-5.738, 28.157],[-5.965, 27.704],[-6.166, 27.207],[-6.325, 26.698],
]

start_id = 5050
segment_node_ids = []

for i, (x, y) in enumerate(slave_node_setA):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["slave_nodes_setA"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [71]:
slaveNode_A = block_registry["slave_nodes_setA"]["nodes"]
print(master_node_ids)

[5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010, 5011, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5027, 5028, 5029, 5030, 5031, 5032, 5033, 5034, 5035, 5036, 5037, 5038, 5039, 5040]


In [72]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

5067


In [73]:
slave_node_setB = [
[-6.388, 23.552],
[-6.258, 23.068],
[-6.076, 22.555],
[-5.874, 22.102],
[-5.620, 21.636],
[-5.335, 21.200],
[-5.040, 20.817],
[-4.685, 20.423],
[-4.299, 20.058],
[-3.890, 19.730],
[-3.452, 19.433],
[-3.040, 19.198],
[-2.601, 18.988],
[-2.083, 18.790],
[-1.608, 18.650],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(slave_node_setB):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["slave_nodes_setB"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [74]:
slaveNode_B = block_registry["slave_nodes_setB"]["nodes"]
print(master_node_ids)

[5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010, 5011, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5027, 5028, 5029, 5030, 5031, 5032, 5033, 5034, 5035, 5036, 5037, 5038, 5039, 5040]


In [75]:
final_slave_nodes = slaveNode_A + [3038, 2990, 2942, 2894, 2846] + slaveNode_B + [2316, 2317, 2318]
print(final_slave_nodes)

[5050, 5051, 5052, 5053, 5054, 5055, 5056, 5057, 5058, 5059, 5060, 5061, 5062, 5063, 5064, 5065, 5066, 3038, 2990, 2942, 2894, 2846, 5067, 5068, 5069, 5070, 5071, 5072, 5073, 5074, 5075, 5076, 5077, 5078, 5079, 5080, 5081, 2316, 2317, 2318]


In [76]:
# Inputs
lagrange_start_id = 5100
num_lagrange_nodes = 40
x_coord = 0.0
y_coord = 18.7

# Output list
lagrange_node_ids = []

# Create nodes
for i in range(num_lagrange_nodes):
    nid = lagrange_start_id + i
    ops.node(nid, x_coord, y_coord)
    created_nodes.append((nid, x_coord, y_coord))
    lagrange_node_ids.append(nid)

# Register block
block_registry["lagrange_nodes"] = {
    "nodes": lagrange_node_ids,
    "elements": []
}


In [ ]:
lagrange_node_ids = block_registry["lagrange_nodes"]["nodes"]
print(lagrange_node_ids)

[5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010, 5011, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5027, 5028, 5029, 5030, 5031, 5032, 5033, 5034, 5035, 5036, 5037, 5038, 5039, 5040]


In [78]:
ops.model("basic", "-ndm", 2, "-ndf", 3)

In [79]:
master_nodes = master_node_ids
slave_nodes = final_slave_nodes
lagrange_nodes = lagrange_node_ids

contact_elem_ids = []
beam_start_id = 4300
for i in range(len(slave_nodes)):
    tag = beam_start_id + i
    iN = master_nodes[i]
    jN = master_nodes[i + 1]
    sN = slave_nodes[i]
    IN = lagrange_nodes[i]

    ops.element("BeamContact2D", tag, iN, jN, sN, IN, contact_mat, 0.5, 1e-10, 1e-10)
    created_elements.append((tag, iN, jN, sN, IN))
    contact_elem_ids.append(tag)

block_registry["contact_beams"] = {
    "nodes": master_nodes + slave_nodes + lagrange_nodes,
    "elements": contact_elem_ids
}

In [ ]:
# Inputs
beam_nodes = [100, 99, 98, 97]  # Example node list (ordered start → end)
transFTag = 1
beam_secTag = 1
intTag = 401
Nint = 3
beam_start_id = 1000  # Starting element tag


# Geometry transformation and section definition
ops.geomTransf("Linear", transFTag)
ops.section("Elastic", beam_secTag, 200e6, 0.5, 0.000975)
ops.beamIntegration("Legendre", intTag, beam_secTag, Nint)

# Create elements
beam_elem_ids = []
for i in range(len(beam_nodes) - 1):
    sN = beam_nodes[i]
    eN = beam_nodes[i + 1]
    eid = beam_start_id + i
    ops.element("dispBeamColumn", eid, sN, eN, transFTag, intTag)
    created_elements.append((eid, sN, eN))
    beam_elem_ids.append(eid)

# Register block
block_registry["beam_elements"] = {
    "nodes": beam_nodes,
    "elements": beam_elem_ids
}

In [ ]:
opst.vis.plotly.plot_model()

In [ ]:
# Print node details
print("Nodes in OAE_Soil_block:")
for nid in block_registry["OAE_Soil_block"]["nodes"]:
    x, y = ops.nodeCoord(nid)
    print(f"Node {nid}: x = {x:.3f}, y = {y:.3f}")

# Print element details
print("\nElements in OAE_Soil_block:")
for eid in block_registry["OAE_Soil_block"]["elements"]:
    node_ids = ops.eleNodes(eid)
    print(f"Element {eid}: nodes = {node_ids}")


Nodes in OAE_Soil_block:
Node 1: x = -30.000, y = 0.000
Node 2: x = -29.500, y = 0.000
Node 3: x = -29.000, y = 0.000
Node 4: x = -28.500, y = 0.000
Node 5: x = -28.000, y = 0.000
Node 6: x = -27.500, y = 0.000
Node 7: x = -27.000, y = 0.000
Node 8: x = -26.500, y = 0.000
Node 9: x = -26.000, y = 0.000
Node 10: x = -25.500, y = 0.000
Node 11: x = -25.000, y = 0.000
Node 12: x = -24.500, y = 0.000
Node 13: x = -24.000, y = 0.000
Node 14: x = -23.500, y = 0.000
Node 15: x = -23.000, y = 0.000
Node 16: x = -22.500, y = 0.000
Node 17: x = -22.000, y = 0.000
Node 18: x = -21.500, y = 0.000
Node 19: x = -21.000, y = 0.000
Node 20: x = -20.500, y = 0.000
Node 21: x = -20.000, y = 0.000
Node 22: x = -19.500, y = 0.000
Node 23: x = -19.000, y = 0.000
Node 24: x = -18.500, y = 0.000
Node 25: x = -18.000, y = 0.000
Node 26: x = -17.500, y = 0.000
Node 27: x = -17.000, y = 0.000
Node 28: x = -16.500, y = 0.000
Node 29: x = -16.000, y = 0.000
Node 30: x = -15.500, y = 0.000
Node 31: x = -15.000, y 

In [ ]:
# Nodes in specific block
OAE_node_ids = block_registry["OAE_Soil_block"]["nodes"]
#soil_nodes = [(nid, x, y) for (nid, x, y) in created_nodes if nid in soil_node_ids]

# Elements in specific block
OAE_element_ids = block_registry["Fill_Soil_block"]["elements"]
#soil_elements = [(eid, n1, n2, n3, n4) for (eid, n1, n2, n3, n4) in created_elements if eid in soil_element_ids]
interface_node_at_segmentids = block_registry["interface_nodes_at_lining"]["nodes"]
print(OAE_node_ids)
print(OAE_element_ids)
print(interface_node_at_segmentids)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22

### Plot Nodes

In [ ]:
import openseespy.opensees as ops
import plotly.graph_objects as go

# Get all node tags and coordinates
node_tags = ops.getNodeTags()
node_coords = [ops.nodeCoord(tag) for tag in node_tags]

# Prepare data for plot
x_coords = [coord[0] for coord in node_coords]
y_coords = [coord[1] for coord in node_coords]
labels = [str(tag) for tag in node_tags]

# Plot
fig = go.Figure(data=go.Scatter(
    x=x_coords,
    y=y_coords,
    mode='markers+text',
    text=labels,
    textposition="top center",
    marker=dict(size=6, color='blue')
))

fig.update_layout(
    title="OpenSees Node Plot",
    xaxis_title="X",
    yaxis_title="Y",
    xaxis=dict(scaleanchor="y", scaleratio=1),
    height=1000,
    width=1500,
    showlegend=False
)

fig.show()
